# Refuerzo: Fundamentos de Regresión — California Housing
## Ingeniería del Conocimiento (ISO56B) — UNCP
### Msc. Jaime Antonio Huaytalla Pariona — Semestre 2026-II

**Objetivo:** Comprender paso a paso los conceptos de regresión en Machine Learning, desde la exploración de datos hasta la evaluación de modelos básicos.

**Nivel:** Básico → Intermedio

**Contenido:**
1. ¿Qué es la regresión? Exploración visual del dataset
2. Train/Test Split
3. Regresión Lineal Simple y Múltiple
4. Métricas de evaluación: MSE, RMSE, MAE, R²
5. Regresión Polinomial y overfitting
6. Árbol de Decisión para regresión
7. Comparación de modelos


## 1. Configuración del Entorno

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

# Configuración de gráficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
np.random.seed(42)

print("✅ Librerías cargadas correctamente")


## 2. ¿Qué es la Regresión? — Exploración del Dataset

La **regresión** es una tarea de aprendizaje supervisado donde predecimos un **valor numérico continuo** (no una categoría).

**Dataset California Housing:**
- 20,640 bloques de viviendas en California
- 8 features numéricos (ingreso medio, antigüedad, habitaciones, etc.)
- **Target:** precio medio de la vivienda (en cientos de miles de dólares)

Nuestro objetivo: predecir el precio de una vivienda a partir de sus características.


In [ ]:
# Cargar dataset
data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['MedHouseVal'] = data.target  # Target: precio medio

print(f"Dimensiones: {df.shape[0]} muestras × {df.shape[1]-1} features + 1 target")
print(f"\nFeatures: {list(data.feature_names)}")
print(f"\nTarget: MedHouseVal (precio medio en $100,000)")
print(f"\nPrimeras 5 filas:")
df.head()


In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas del dataset:")
df.describe().round(3)


### 2.1 Visualización: ¿Cómo se distribuye el target?

Antes de modelar, debemos entender la distribución de lo que queremos predecir.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma del target
axes[0].hist(df['MedHouseVal'], bins=50, color='#2E86AB', edgecolor='white', alpha=0.8)
axes[0].axvline(df['MedHouseVal'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Media: {df["MedHouseVal"].mean():.2f}')
axes[0].axvline(df['MedHouseVal'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Mediana: {df["MedHouseVal"].median():.2f}')
axes[0].set_xlabel('Precio medio ($100K)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del Precio de Viviendas')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['MedHouseVal'], vert=True)
axes[1].set_ylabel('Precio medio ($100K)')
axes[1].set_title('Boxplot del Precio')

plt.tight_layout()
plt.show()

print(f"Nota: hay un tope en $5.0 (=$500,000). Valores en ese límite son censurados.")


### 2.2 Relación entre Features y Target

Veamos cómo cada feature se relaciona con el precio. Las **correlaciones** nos dan una primera idea.


In [ ]:
# Matriz de correlación
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Matriz de Correlación — California Housing')
plt.tight_layout()
plt.show()

print("\nCorrelaciones con el precio (MedHouseVal):")
print(corr['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False).to_string())


In [ ]:
# Scatter plots de las 4 features más correlacionadas con el target
top_features = ['MedInc', 'AveRooms', 'HouseAge', 'AveOccup']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.ravel(), top_features):
    ax.scatter(df[feat], df['MedHouseVal'], alpha=0.1, s=5, color='#2E86AB')
    ax.set_xlabel(feat)
    ax.set_ylabel('MedHouseVal')
    ax.set_title(f'{feat} vs Precio')

plt.suptitle('Relación Feature → Precio', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observación: MedInc (ingreso medio) tiene la correlación más fuerte con el precio.")
print("Esto tiene sentido: zonas con mayores ingresos tienen casas más caras.")


## 3. Train/Test Split

Dividimos los datos ANTES de cualquier preprocesamiento para evitar **data leakage** (fuga de información).


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Datos de entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Datos de prueba:        {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\n¿Por qué dividir?")
print("→ Para evaluar cómo se comporta el modelo con datos que NUNCA vio durante el entrenamiento.")
print("→ Si evaluamos con los mismos datos de entrenamiento, no sabremos si memoriza o aprende.")


## 4. Primer Modelo: Regresión Lineal

### 4.1 Regresión Lineal Simple (1 feature)

Empecemos con lo más simple: usar SOLO el ingreso medio (MedInc) para predecir el precio. El modelo busca la **mejor recta** que relaciona ambos.

**Modelo: ŷ = w₀ + w₁ · MedInc**


In [ ]:
from sklearn.linear_model import LinearRegression

# Regresión lineal con 1 solo feature
X_train_simple = X_train[['MedInc']]
X_test_simple = X_test[['MedInc']]

lr_simple = LinearRegression()
lr_simple.fit(X_train_simple, y_train)

print(f"Modelo: ŷ = {lr_simple.intercept_:.4f} + {lr_simple.coef_[0]:.4f} × MedInc")
print(f"\nInterpretación:")
print(f"  Intercepto = {lr_simple.intercept_:.4f} → precio base cuando MedInc = 0")
print(f"  Pendiente  = {lr_simple.coef_[0]:.4f} → por cada unidad más de ingreso medio,")
print(f"                el precio sube ${lr_simple.coef_[0]*100000:.0f}")


In [ ]:
# Visualizar la recta de regresión
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(X_test_simple, y_test, alpha=0.2, s=10, color='#2E86AB', label='Datos reales')

# Línea de regresión
x_line = np.linspace(X_test_simple.min().values[0], X_test_simple.max().values[0], 100)
y_line = lr_simple.predict(x_line.reshape(-1, 1))
ax.plot(x_line, y_line, color='red', linewidth=2, label='Recta de regresión')

ax.set_xlabel('MedInc (ingreso medio)')
ax.set_ylabel('MedHouseVal (precio)')
ax.set_title('Regresión Lineal Simple: MedInc → Precio')
ax.legend()
plt.tight_layout()
plt.show()

r2_simple = lr_simple.score(X_test_simple, y_test)
print(f"R² con 1 feature (MedInc): {r2_simple:.4f}")
print(f"→ El ingreso medio solo explica el {r2_simple*100:.1f}% de la variación del precio")


### 4.2 Regresión Lineal Múltiple (todos los features)

Ahora usamos TODOS los features. El modelo busca la combinación lineal óptima:

**ŷ = w₀ + w₁·MedInc + w₂·HouseAge + w₃·AveRooms + ... + w₈·Longitude**


In [ ]:
# Regresión lineal con todos los features
lr_multi = LinearRegression()
lr_multi.fit(X_train, y_train)

print("Coeficientes del modelo:")
print(f"  Intercepto: {lr_multi.intercept_:.4f}")
for name, coef in sorted(zip(X.columns, lr_multi.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {name:15s}: {coef:+.4f}")

r2_multi = lr_multi.score(X_test, y_test)
print(f"\nR² con todos los features: {r2_multi:.4f}")
print(f"R² con solo MedInc:        {r2_simple:.4f}")
print(f"\n→ Usar todos los features mejora el R² de {r2_simple:.4f} a {r2_multi:.4f}")


## 5. Métricas de Evaluación para Regresión

En clasificación usamos accuracy, precision, recall. En regresión medimos **qué tan lejos** están las predicciones de los valores reales.


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = lr_multi.predict(X_test)

# Calcular todas las métricas
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("=" * 55)
print("    MÉTRICAS DE EVALUACIÓN — Regresión Lineal Múltiple")
print("=" * 55)
print(f"  MSE  = {mse:.4f}  (Error Cuadrático Medio)")
print(f"  RMSE = {rmse:.4f}  (Raíz del MSE → mismas unidades que y)")
print(f"  MAE  = {mae:.4f}  (Error Absoluto Medio)")
print(f"  R²   = {r2:.4f}   (Coeficiente de Determinación)")
print("=" * 55)
print(f"\nInterpretación práctica:")
print(f"  • RMSE = {rmse:.4f} → las predicciones se desvían ~${rmse*100000:.0f} en promedio")
print(f"  • MAE  = {mae:.4f} → el error medio absoluto es ~${mae*100000:.0f}")
print(f"  • R²   = {r2:.4f} → el modelo explica el {r2*100:.1f}% de la variabilidad")


### 5.1 Visualización de Predicciones vs Valores Reales

Un buen modelo debería tener puntos cercanos a la diagonal (predicción = real).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Predicción vs Real
axes[0].scatter(y_test, y_pred, alpha=0.2, s=10, color='#2E86AB')
axes[0].plot([0, 5], [0, 5], 'r--', linewidth=2, label='Predicción perfecta')
axes[0].set_xlabel('Valor Real')
axes[0].set_ylabel('Predicción')
axes[0].set_title('Predicción vs Real')
axes[0].legend()

# 2. Residuos vs Predicción
residuos = y_test - y_pred
axes[1].scatter(y_pred, residuos, alpha=0.2, s=10, color='#A23B72')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Residuo (real − predicción)')
axes[1].set_title('Gráfico de Residuos')

# 3. Distribución de residuos
axes[2].hist(residuos, bins=50, color='#2E86AB', edgecolor='white', alpha=0.8)
axes[2].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residuo')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribución de Residuos')

plt.suptitle('Análisis de Residuos — Regresión Lineal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("¿Qué buscar?")
print("• Pred vs Real: puntos cerca de la diagonal = buen modelo")
print("• Residuos: distribución aleatoria alrededor de 0 = modelo sin sesgo")
print("• Histograma: forma de campana centrada en 0 = errores normales")


## 6. Regresión Polinomial y el Problema del Overfitting

¿Qué pasa si la relación no es una línea recta? Podemos crear **features polinómicas**.

Veamos esto con un ejemplo simple: predecir el precio usando solo MedInc con grados 1, 3 y 15.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Usar solo MedInc para visualización clara
X_tr_s = X_train[['MedInc']].values
X_te_s = X_test[['MedInc']].values

degrees = [1, 3, 15]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, deg in zip(axes, degrees):
    # Pipeline: PolyFeatures → LinearRegression
    pipe = make_pipeline(PolynomialFeatures(deg, include_bias=False), LinearRegression())
    pipe.fit(X_tr_s, y_train)

    train_r2 = pipe.score(X_tr_s, y_train)
    test_r2  = pipe.score(X_te_s, y_test)

    ax.scatter(X_te_s, y_test, alpha=0.1, s=5, color='#2E86AB')
    x_plot = np.linspace(0, 16, 300).reshape(-1, 1)
    y_plot = pipe.predict(x_plot)
    # Limitar y_plot para visualización
    y_plot = np.clip(y_plot, -1, 7)
    ax.plot(x_plot, y_plot, color='red', linewidth=2)
    ax.set_ylim(-0.5, 6)
    ax.set_xlabel('MedInc')
    ax.set_ylabel('Precio')
    ax.set_title(f'Grado {deg}\nTrain R²={train_r2:.3f} | Test R²={test_r2:.3f}')

plt.suptitle('Efecto del Grado Polinomial (Overfitting vs Underfitting)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observaciones:")
print("• Grado 1 (lineal): captura la tendencia general pero pierde la curvatura")
print("• Grado 3: mejor ajuste, captura no linealidad sin sobreajustar")
print("• Grado 15: se ajusta demasiado al ruido → OVERFITTING")
print("  (Train R² alto pero Test R² puede bajar)")


## 7. Árbol de Decisión para Regresión

Los árboles no buscan una ecuación lineal. Dividen el espacio de features en **regiones** y predicen el **promedio** de los targets en cada región.


In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Probar diferentes profundidades
depths = [2, 5, 10, None]
results = []

for d in depths:
    tree = DecisionTreeRegressor(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    tr_r2 = tree.score(X_train, y_train)
    te_r2 = tree.score(X_test, y_test)
    results.append({'max_depth': str(d), 'R² Train': tr_r2, 'R² Test': te_r2})
    print(f"max_depth={str(d):5s} → Train R²={tr_r2:.4f} | Test R²={te_r2:.4f}")

print("\nObservaciones:")
print("• Profundidad baja → modelo simple, posible underfitting")
print("• Profundidad alta o None → el árbol memoriza → OVERFITTING")
print("• Buscar el punto donde Test R² es máximo")


In [ ]:
# Curva de overfitting: profundidad vs R²
max_depths = range(1, 25)
train_scores = []
test_scores = []

for d in max_depths:
    tree = DecisionTreeRegressor(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(max_depths, train_scores, 'o-', color='#2E86AB', label='Train R²', linewidth=2)
ax.plot(max_depths, test_scores, 's-', color='#A23B72', label='Test R²', linewidth=2)
ax.axvline(x=max_depths[np.argmax(test_scores)], color='green', linestyle='--',
           label=f'Óptimo: depth={max_depths[np.argmax(test_scores)]}')
ax.set_xlabel('max_depth')
ax.set_ylabel('R²')
ax.set_title('Curva de Overfitting — Árbol de Decisión')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mejor profundidad: {max_depths[np.argmax(test_scores)]} (Test R² = {max(test_scores):.4f})")


In [ ]:
# Visualizar el árbol óptimo (primeros niveles)
from sklearn.tree import plot_tree

best_depth = max_depths[np.argmax(test_scores)]
tree_best = DecisionTreeRegressor(max_depth=min(best_depth, 4), random_state=42)
tree_best.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(tree_best, feature_names=data.feature_names, filled=True,
          rounded=True, fontsize=8, ax=ax, max_depth=3)
ax.set_title(f'Árbol de Decisión (mostrando primeros 3 niveles)', fontsize=14)
plt.tight_layout()
plt.show()

print("Cada nodo muestra:")
print("• La regla de decisión (ej: MedInc <= 3.07)")
print("• El MSE de las muestras en ese nodo")
print("• El número de muestras")
print("• El valor predicho (promedio de las muestras)")


## 8. Comparación de Modelos

Comparemos todos los modelos que hemos visto usando las mismas métricas.


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Modelos a comparar
modelos = {
    'Lineal Simple (MedInc)': LinearRegression().fit(X_train[['MedInc']], y_train),
    'Lineal Múltiple':        LinearRegression().fit(X_train, y_train),
    'Polinomial (grado 3)':   make_pipeline(PolynomialFeatures(3, include_bias=False),
                                             LinearRegression()).fit(X_train, y_train),
    'Árbol (depth=best)':     DecisionTreeRegressor(max_depth=best_depth,
                                                     random_state=42).fit(X_train, y_train),
}

print(f"{'Modelo':<28s} {'R²':>8s} {'RMSE':>8s} {'MAE':>8s}")
print("=" * 56)

results_list = []
for nombre, modelo in modelos.items():
    if 'Simple' in nombre:
        yp = modelo.predict(X_test[['MedInc']])
    else:
        yp = modelo.predict(X_test)
    r2   = r2_score(y_test, yp)
    rmse = np.sqrt(mean_squared_error(y_test, yp))
    mae  = mean_absolute_error(y_test, yp)
    results_list.append({'Modelo': nombre, 'R²': r2, 'RMSE': rmse, 'MAE': mae})
    print(f"{nombre:<28s} {r2:8.4f} {rmse:8.4f} {mae:8.4f}")

print("\n💡 Conclusión: más features y modelos más flexibles mejoran la predicción,")
print("   pero hay que cuidar el overfitting (el árbol sin límite puede memorizar).")


In [ ]:
# Gráfico comparativo
df_results = pd.DataFrame(results_list)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ['#2E86AB', '#A23B72', '#F18F01', '#1B5E20']

for ax, metric in zip(axes, ['R²', 'RMSE', 'MAE']):
    bars = ax.barh(df_results['Modelo'], df_results[metric], color=colors)
    ax.set_xlabel(metric)
    ax.set_title(metric)
    for bar, val in zip(bars, df_results[metric]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)

plt.suptitle('Comparación de Modelos de Regresión', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Resumen y Próximos Pasos

### Lo que aprendimos:
| Concepto | Detalle |
|----------|---------|
| **Regresión** | Predecir un valor numérico continuo |
| **Regresión Lineal** | Ajustar la mejor recta/hiperplano a los datos |
| **Métricas** | MSE, RMSE (mismas unidades), MAE (robusto), R² (0-1) |
| **Overfitting** | El modelo memoriza en vez de aprender (Polynomial grado alto, Árbol profundo) |
| **Train/Test Split** | Evaluar siempre con datos que el modelo NO vio |

### En el siguiente notebook:
- Estandarización y Pipelines
- Regularización: Ridge, Lasso, Elastic Net
- Modelos de ensamble: Random Forest, Gradient Boosting
- Validación Cruzada y GridSearchCV
- Importancia de features

---
*Material de refuerzo — Ingeniería del Conocimiento (ISO56B) — UNCP*
